In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [ ]:
!pip install torch-geometric
!pip install rdkit
!pip install -U bitsandbytes


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import torch.nn.functional as F
import os

# Data loading

In [ ]:
from google.colab import drive
import os
import pandas as pd
from pathlib import Path
from google.colab import drive

# 1. Monter Google Drive
drive.mount('/content/drive')

# --- CONFIGURATION DES CHEMINS ---
# Ajustez 'Mon_Projet' selon le nom de votre dossier dans Drive
base_drive_path = Path("/content/drive/MyDrive/Altegrad")



In [ ]:
batch_size = 32
num_epochs = 20
learning_rate = 2e-5
weight_decay = 0.01
warmup_ratio = 0.1
max_smiles_len = 256
max_text_len = 128
temperature = 0.07
num_workers = 4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
import pickle
import torch

# 1. Load the data

file_path = base_drive_path / 'train_graphs.pkl'

print(f"Loading {file_path}...")
with open(file_path, 'rb') as f:
    graphs = pickle.load(f)

print(f"Successfully loaded {len(graphs)} molecules.\n")

# 2. Inspect the first molecule (index 0)
data = graphs[0]

print("--- Overview of the first Molecule ---")
print(f"Object type: {type(data)}")
print(f"Keys available: {data.keys}\n")

# 3. Inspect specific attributes
print(f"ID: {data.id}")
print(f"Description: {data.description}")

print(f"\nNode Features (data.x):")
print(f"  Shape: {data.x.shape} (Atoms x Features)")
print(f"  First atom features: {data.x[0]}")

print(f"\nConnectivity (data.edge_index):")
print(f"  Shape: {data.edge_index.shape}")
print(f"  Structure (Source -> Target):\n{data.edge_index}")

if hasattr(data, 'edge_attr') and data.edge_attr is not None:
    print(f"\nBond Features (data.edge_attr):")
    print(f"  Shape: {data.edge_attr.shape}")

In [ ]:
import networkx as nx
from torch_geometric.utils import to_networkx
import matplotlib.pyplot as plt

# Convert PyG Data object to a NetworkX graph
# to_undirected=True is usually better for visualization of molecules
G = to_networkx(data, to_undirected=True)

plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G, seed=42)  # Compute layout for clear visualization

# Draw the nodes and edges
nx.draw(G, pos,
        with_labels=True,
        node_color='lightblue',
        node_size=500,
        font_size=10,
        edge_color='gray')

plt.title(f"Graph Structure for Molecule ID: {data.id}")
plt.show()

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
import torch

def pyg_to_mol(data):
    """
    Converts a PyTorch Geometric Data object to an RDKit Molecule.
    Assumes data.x[:, 0] contains atomic numbers.
    """
    # 1. Create an empty editable molecule
    mol = Chem.RWMol()

    # 2. Add Atoms
    # We iterate through the node features (data.x)
    atom_indices = []
    for atom_features in data.x:
        # Assumption: The first feature is the Atomic Number (e.g., 6=C, 8=O)
        # We accept float but convert to int
        atomic_num = int(atom_features[0].item())

        # Create RDKit atom and add to molecule
        if atomic_num > 0: # minimal check for valid atoms
            a = Chem.Atom(atomic_num)
            idx = mol.AddAtom(a)
            atom_indices.append(idx)
        else:
            # Fallback for dummy atoms if 0 is used
            a = Chem.Atom(0)
            idx = mol.AddAtom(a)
            atom_indices.append(idx)

    # 3. Add Bonds
    # data.edge_index is shape [2, Num_Edges]
    rows, cols = data.edge_index

    # We loop through edges.
    # Note: PyG graphs are often directed (edges both ways), RDKit needs undirected.
    # We use a set to avoid adding the same bond twice.
    seen_bonds = set()

    for i in range(len(rows)):
        src = int(rows[i].item())
        dst = int(cols[i].item())

        # Sort to handle undirected nature (0-1 is same as 1-0)
        bond_sig = tuple(sorted((src, dst)))

        if bond_sig not in seen_bonds and src != dst:
            mol.AddBond(src, dst, Chem.BondType.SINGLE)
            seen_bonds.add(bond_sig)

    # 4. Sanitize (Detect aromatic rings, valence, etc.)
    try:
        Chem.SanitizeMol(mol)
    except ValueError as e:
        print(f"Warning: Molecule sanitization failed (likely invalid valences): {e}")

    return mol.GetMol()

In [ ]:
# Select a molecule (e.g., index 0)
data = graphs[0]

# Convert to RDKit
mol = pyg_to_mol(data)

# Display
print(f"Molecule ID: {data.id}")
print(f"Description: {data.description}")

# This will render the image in the notebook
display(Draw.MolToImage(mol, size=(400, 300)))

## Preprocessing

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdchem
import torch

def data_to_rdkit_mol(data, x_map, e_map):
    """
    data: torch_geometric.data.Data
    x_map: dict of categorical mappings for node features (from data_utils.py)
    e_map: dict of categorical mappings for edge features (from data_utils.py)
    """

    mol = Chem.RWMol()

    # ---- 1) Add atoms ----
    for i in range(data.x.size(0)):
        feats = data.x[i]

        # Indices in feats must match the order described in your statement:
        # [atomic_num, chirality, degree, formal_charge, num_hs,
        #  num_radical_electrons, hybridization, is_aromatic, is_in_ring]
        atomic_num_idx        = int(feats[0])
        chirality_idx         = int(feats[1])
        degree_idx            = int(feats[2])
        formal_charge_idx     = int(feats[3])
        num_hs_idx            = int(feats[4])
        num_rad_e_idx         = int(feats[5])
        hybridization_idx     = int(feats[6])
        is_aromatic_idx       = int(feats[7])
        is_in_ring_idx        = int(feats[8])

        atomic_num    = x_map["atomic_num"][atomic_num_idx]
        formal_charge = x_map["formal_charge"][formal_charge_idx]
        num_hs        = x_map["num_hs"][num_hs_idx]
        num_rad_e     = x_map["num_radical_electrons"][num_rad_e_idx]

        # RDKit atom
        atom = Chem.Atom(int(atomic_num))
        atom.SetFormalCharge(int(formal_charge))
        atom.SetNumExplicitHs(int(num_hs))
        atom.SetNumRadicalElectrons(int(num_rad_e))

        # Hybridization
        hybridization_str = x_map["hybridization"][hybridization_idx]
        hyb_map = {
            "S":   rdchem.HybridizationType.S,
            "SP":  rdchem.HybridizationType.SP,
            "SP2": rdchem.HybridizationType.SP2,
            "SP3": rdchem.HybridizationType.SP3,
            "SP3D": rdchem.HybridizationType.SP3D,
            "SP3D2": rdchem.HybridizationType.SP3D2,
            "UNSPECIFIED": rdchem.HybridizationType.UNSPECIFIED,
            "OTHER": rdchem.HybridizationType.UNSPECIFIED,
        }
        atom.SetHybridization(hyb_map.get(hybridization_str, rdchem.HybridizationType.UNSPECIFIED))

        # Chirality
        chiral_str = x_map["chirality"][chirality_idx]
        chiral_map = {
            "CHI_UNSPECIFIED": rdchem.ChiralType.CHI_UNSPECIFIED,
            "CHI_TETRAHEDRAL_CW": rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
            "CHI_TETRAHEDRAL_CCW": rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,
        }
        atom.SetChiralTag(chiral_map.get(chiral_str, rdchem.ChiralType.CHI_UNSPECIFIED))

        # Aromaticity
        is_aromatic = bool(x_map["is_aromatic"][is_aromatic_idx])
        atom.SetIsAromatic(is_aromatic)

        mol_idx = mol.AddAtom(atom)

    # ---- 2) Add bonds ----
    edge_index = data.edge_index
    edge_attr  = data.edge_attr

    # Here we avoid adding each bond twice (because edges are i->j and j->i)
    added_bonds = set()

    for k in range(edge_index.size(1)):
        u = int(edge_index[0, k])
        v = int(edge_index[1, k])
        if u == v:
            continue

        key = tuple(sorted((u, v)))
        if key in added_bonds:
            continue
        added_bonds.add(key)

        bf = edge_attr[k]
        bond_type_idx   = int(bf[0])
        stereo_idx      = int(bf[1])
        is_conj_idx     = int(bf[2])

        bond_type_str = e_map["bond_type"][bond_type_idx]
        stereo_str    = e_map["stereo"][stereo_idx]
        is_conjugated = bool(e_map["is_conjugated"][is_conj_idx])

        # Bond type mapping (adapt to your e_map)
        bond_type_map = {
            "SINGLE": rdchem.BondType.SINGLE,
            "DOUBLE": rdchem.BondType.DOUBLE,
            "TRIPLE": rdchem.BondType.TRIPLE,
            "AROMATIC": rdchem.BondType.AROMATIC,
        }
        bt = bond_type_map.get(bond_type_str, rdchem.BondType.SINGLE)

        mol.AddBond(u, v, bt)
        bond = mol.GetBondBetweenAtoms(u, v)
        if bond is None:
            continue

        bond.SetIsConjugated(is_conjugated)

        stereo_map = {
            "StereoNone": rdchem.BondStereo.STEREONONE,
            "StereoAny": rdchem.BondStereo.STEREOANY,
            "StereoZ": rdchem.BondStereo.STEREOZ,
            "StereoE": rdchem.BondStereo.STEREOE,
            "StereoCis": rdchem.BondStereo.STEREOZ,
            "StereoTrans": rdchem.BondStereo.STEREOE,
        }
        bond.SetStereo(stereo_map.get(stereo_str, rdchem.BondStereo.STEREONONE))

    # ---- 3) Finalize molecule ----
    mol = mol.GetMol()
    Chem.SanitizeMol(mol)
    return mol


def data_to_smiles(data, x_map, e_map):
    mol = data_to_rdkit_mol(data, x_map, e_map)
    # canonical SMILES
    return Chem.MolToSmiles(mol)


In [ ]:
graphs

In [ ]:
"""
Data loading and processing utilities for molecule-text retrieval.
Includes dataset classes and data loading functions.
"""
from typing import Dict
import pickle

import pandas as pd
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Batch


# =========================================================
# Feature maps for atom and bond attributes
# =========================================================
from typing import Dict, List, Any

x_map: Dict[str, List[Any]] = {
    'atomic_num': list(range(0, 119)),
    'chirality': [
        'CHI_UNSPECIFIED','CHI_TETRAHEDRAL_CW','CHI_TETRAHEDRAL_CCW','CHI_OTHER',
        'CHI_TETRAHEDRAL','CHI_ALLENE','CHI_SQUAREPLANAR','CHI_TRIGONALBIPYRAMIDAL',
        'CHI_OCTAHEDRAL',
    ],
    'degree': list(range(0, 11)),
    'formal_charge': list(range(-5, 7)),
    'num_hs': list(range(0, 9)),
    'num_radical_electrons': list(range(0, 5)),
    'hybridization': [
        'UNSPECIFIED','S','SP','SP2','SP3','SP3D','SP3D2','OTHER',
    ],
    'is_aromatic': [False, True],
    'is_in_ring': [False, True],
}

e_map: Dict[str, List[Any]] = {
    'bond_type': [
        'UNSPECIFIED','SINGLE','DOUBLE','TRIPLE','QUADRUPLE','QUINTUPLE','HEXTUPLE',
        'ONEANDAHALF','TWOANDAHALF','THREEANDAHALF','FOURANDAHALF','FIVEANDAHALF',
        'AROMATIC','IONIC','HYDROGEN','THREECENTER','DATIVEONE','DATIVE','DATIVEL',
        'DATIVER','OTHER','ZERO',
    ],
    'stereo': [
        'STEREONONE','STEREOANY','STEREOZ','STEREOE','STEREOCIS','STEREOTRANS',
    ],
    'is_conjugated': [False, True],
}


# =========================================================
# Load precomputed text embeddings
# =========================================================
def load_id2emb(csv_path: str) -> Dict[str, torch.Tensor]:
    """
    Load precomputed text embeddings from CSV file.

    Args:
        csv_path: Path to CSV file with columns: ID, embedding
                  where embedding is comma-separated floats

    Returns:
        Dictionary mapping ID (str) to embedding tensor
    """
    df = pd.read_csv(csv_path)
    id2emb = {}
    for _, row in df.iterrows():
        id_ = str(row["ID"])
        emb_str = row["embedding"]
        emb_vals = [float(x) for x in str(emb_str).split(',')]
        id2emb[id_] = torch.tensor(emb_vals, dtype=torch.float32)
    return id2emb


# =========================================================
# Load descriptions from preprocessed graphs
# =========================================================
def load_descriptions_from_graphs(graph_path: str) -> Dict[str, str]:
    """
    Load ID to description mapping from preprocessed graph file.

    Args:
        graph_path: Path to .pkl file containing list of pre-saved graphs

    Returns:
        Dictionary mapping ID (str) to description (str)
    """
    with open(graph_path, 'rb') as f:
        graphs = pickle.load(f)

    id2desc = {}
    for graph in graphs:
        id2desc[graph.id] = graph.description

    return id2desc


# =========================================================
# Dataset that loads preprocessed graphs and text embeddings
# =========================================================
class PreprocessedGraphDataset(Dataset):
    """
    Dataset that loads pre-saved molecule graphs with optional text embeddings.

    Args:
        graph_path: Path to .pkl file containing list of pre-saved graphs
        emb_dict: Dictionary mapping ID to text embedding tensors (optional)
    """
    def __init__(self, graph_path: str, emb_dict: Dict[str, torch.Tensor] = None):
        print(f"Loading graphs from: {graph_path}")
        with open(graph_path, 'rb') as f:
            self.graphs = pickle.load(f)
        self.emb_dict = emb_dict
        self.ids = [g.id for g in self.graphs]
        print(f"Loaded {len(self.graphs)} graphs")

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, idx):
        graph = self.graphs[idx]
        if self.emb_dict is not None:
            id_ = graph.id
            text_emb = self.emb_dict[id_]
            return graph, text_emb
        else:
            return graph






# Selfies

In [ ]:
pip install selfies

In [ ]:
import selfies as sf
from rdkit import Chem

# 1. RELAX CONSTRAINTS
# This fixes the "input violates... semantic constraints" error (e.g. for Iodine)
# 'hypervalent' allows atoms like I, S, P to have more bonds than usual.
sf.set_semantic_constraints('hypervalent')

def data_to_selfies(data, x_map, e_map):
    # 1. Convert Graph to SMILES
    try:
        smiles = data_to_smiles(data, x_map, e_map)
    except Exception:
        return None

    # 2. Convert SMILES to SELFIES
    try:
        # We allow the encoder to accept the SMILES.
        # Note: If kekulization fails here, it means the molecule is
        # physically ambiguous in the RDKit/SELFIES model.
        return sf.encoder(smiles)
    except Exception as e:
        # Optional: Try to sanitize the SMILES one last time if kekulization failed
        # This sometimes fixes minor aromaticity issues
        try:
            mol = Chem.MolFromSmiles(smiles, sanitize=False)
            if mol:
                Chem.SanitizeMol(mol, sanitizeOps=Chem.SanitizeFlags.SANITIZE_ALL^Chem.SanitizeFlags.SANITIZE_KEKULIZE)
                clean_smiles = Chem.MolToSmiles(mol)
                return sf.encoder(clean_smiles)
        except:
            pass

        # If it still fails (common for complex Organometallics like [Ir], [Ni]),
        # we return None to filter it out.
        return None

# ==========================================
# Run the loop with error tracking
# ==========================================

train_dataset = PreprocessedGraphDataset("/content/drive/MyDrive/Altegrad/train_graphs.pkl")

all_selfies = []
all_ids = []
failed_indices = []

print("Starting conversion...")
for i, data in enumerate(train_dataset):
    selfie = data_to_selfies(data, x_map, e_map)

    if selfie is not None:
        all_selfies.append(selfie)
        all_ids.append(data.id)
    else:
        # Keep track of which indices failed so you can inspect them later
        failed_indices.append(i)

# Create DataFrame
dataset_df = pd.DataFrame({
    "ID": all_ids,
    "selfies": all_selfies,
    # Match descriptions to the successful IDs
    "description": [
        train_dataset.graphs[i].description
        for i in range(len(train_dataset))
        if i not in failed_indices
    ]
})



print(f"Successfully converted: {len(dataset_df)}")
print(f"Failed: {len(failed_indices)}")
print(dataset_df.head())

In [ ]:
val_dataset = PreprocessedGraphDataset("/content/drive/MyDrive/Altegrad/validation_graphs.pkl")

all_selfies = []
all_ids = []
failed_indices = []

print("Starting conversion...")
for i, data in enumerate(val_dataset):
    selfie = data_to_selfies(data, x_map, e_map)

    if selfie is not None:
        all_selfies.append(selfie)
        all_ids.append(data.id)
    else:
        # Keep track of which indices failed so you can inspect them later
        failed_indices.append(i)

# Create DataFrame
dataset_df_val = pd.DataFrame({
    "ID": all_ids,
    "selfies": all_selfies,
    # Match descriptions to the successful IDs
    "description": [
        val_dataset.graphs[i].description
        for i in range(len(val_dataset))
        if i not in failed_indices
    ]
})



print(f"Successfully converted: {len(dataset_df_val)}")
print(f"Failed: {len(failed_indices)}")
print(dataset_df_val.head())

In [ ]:
test_dataset = PreprocessedGraphDataset("/content/drive/MyDrive/Altegrad/test_graphs.pkl")

all_selfies = []
all_ids = []
failed_indices = []

print("Starting conversion...")
for i, data in enumerate(test_dataset):
    selfie = data_to_selfies(data, x_map, e_map)

    if selfie is not None:
        all_selfies.append(selfie)
        all_ids.append(data.id)
    else:
        # Keep track of which indices failed so you can inspect them later
        failed_indices.append(i)

# Create DataFrame
dataset_df_test = pd.DataFrame({
    "ID": all_ids,
    "selfies": all_selfies,
    # Match descriptions to the successful IDs

})



print(f"Successfully converted: {len(dataset_df_test)}")
print(f"Failed: {len(failed_indices)}")
print(dataset_df_test.head())

# Training

In [ ]:
pip install evaluate sacrebleu

In [ ]:
#!pip uninstall -y transformers accelerate
!pip install transformers accelerate sentencepiece

# training

In [ ]:
import os
import shutil
from pathlib import Path
from google.colab import drive
import pandas as pd
from datasets import Dataset
from transformers import (
    TrainerCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)
from transformers.trainer_utils import get_last_checkpoint
import evaluate
import numpy as np

# ==========================================
# 1. GOOGLE DRIVE SETUP
# ==========================================
drive.mount('/content/drive', force_remount=True)

BASE_DRIVE_PATH = Path("/content/drive/MyDrive/Altegrad/BioT5_Experiment")
BASE_DRIVE_PATH.mkdir(parents=True, exist_ok=True)

DRIVE_CHECKPOINT_DIR = BASE_DRIVE_PATH / 'checkpoints'
DRIVE_CHECKPOINT_DIR.mkdir(exist_ok=True)

# ==========================================
# CONFIGURATION
# ==========================================
CONFIG = {
    'model_name': "QizhiPei/biot5-base",
    'batch_size': 32,
    'lr': 5e-5,
    'epochs': 20,
    'max_len': 256,
    'target_len': 256,
    'seed': 42,
    'local_output_dir': "/content/checkpoints",
    'bf16': True
}

class DriveSaveCallback(TrainerCallback):
    """
    After every checkpoint save (locally), copy it to Google Drive.
    """
    def on_save(self, args, state, control, **kwargs):
        # Check if a checkpoint was just saved
        if os.path.isdir(args.output_dir):
            # Find the latest checkpoint folder in local directory
            checkpoints = [d for d in os.listdir(args.output_dir) if d.startswith("checkpoint-")]
            if checkpoints:
                # Sort to get the latest (e.g. checkpoint-500, checkpoint-1000)
                latest_ckpt = sorted(checkpoints, key=lambda x: int(x.split('-')[1]))[-1]
                local_ckpt_path = os.path.join(args.output_dir, latest_ckpt)
                drive_ckpt_path = DRIVE_CHECKPOINT_DIR / latest_ckpt

                # Copy to Drive if it doesn't exist yet
                if not os.path.exists(drive_ckpt_path):
                    print(f"\n[Callback] Syncing {latest_ckpt} to Drive...")
                    shutil.copytree(local_ckpt_path, drive_ckpt_path)
                    print(f"[Callback] Synced.")

def run_training(train_df, val_df):
    os.makedirs(CONFIG['local_output_dir'], exist_ok=True)

    hf_train = Dataset.from_pandas(train_df[["selfies", "description"]])
    hf_val = Dataset.from_pandas(val_df[["selfies", "description"]])

    tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'], model_max_length=CONFIG['max_len'], use_fast=True)
    tokenizer.add_special_tokens({'additional_special_tokens': ["<bom>", "<eom>"]})

    def preprocess_function(examples):
        inputs = [f"Molecule description: <bom>{s}<eom>" for s in examples["selfies"]]
        targets = [d for d in examples["description"]]
        model_inputs = tokenizer(inputs, max_length=CONFIG['max_len'], truncation=True)
        labels = tokenizer(targets, max_length=CONFIG['target_len'], truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    tokenized_train = hf_train.map(preprocess_function, batched=True, num_proc=os.cpu_count())
    tokenized_val = hf_val.map(preprocess_function, batched=True, num_proc=os.cpu_count())

    model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG['model_name'])
    model.resize_token_embeddings(len(tokenizer))

    # --- METRIC SETUP ---
    metric = evaluate.load("sacrebleu")
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple): preds = preds[0]
        preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        decoded_preds = [pred.strip() for pred in decoded_preds]
        decoded_labels = [[label.strip()] for label in decoded_labels]
        result = metric.compute(predictions=decoded_preds, references=decoded_labels)
        return {"bleu": result["score"]}

    # --- TRAINER SETUP ---
    args = Seq2SeqTrainingArguments(
        output_dir=CONFIG['local_output_dir'],
        eval_strategy="epoch",
        save_strategy="epoch", # Save every epoch
        learning_rate=CONFIG['lr'],
        per_device_train_batch_size=CONFIG['batch_size'],
        per_device_eval_batch_size=CONFIG['batch_size'],
        save_total_limit=1,
        num_train_epochs=CONFIG['epochs'],
        predict_with_generate=True,
        fp16=False,
        bf16=CONFIG['bf16'],
        tf32=True,
        dataloader_num_workers=4,
        group_by_length=True,
        logging_dir=str(BASE_DRIVE_PATH / "logs"),
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="tensorboard",
        generation_max_length=CONFIG['target_len'],
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100),
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[DriveSaveCallback]
    )

    # --- RESUME LOGIC ---
    # Check if there is a checkpoint in the Drive folder
    last_checkpoint = get_last_checkpoint(str(DRIVE_CHECKPOINT_DIR))

    print("Starting training...")
    if last_checkpoint:
        print(f"RESUMING from checkpoint found in Drive: {last_checkpoint}")
        trainer.train(resume_from_checkpoint=last_checkpoint)
    else:
        print("No Drive checkpoint found. Starting fresh.")
        trainer.train()

    # Save final model
    print("Saving final model to Drive...")
    final_path = DRIVE_CHECKPOINT_DIR / "final_model_complete"
    trainer.save_model(str(final_path))
    tokenizer.save_pretrained(str(final_path))
    print("Done!")

if __name__ == "__main__":
    run_training(dataset_df, dataset_df_val)

In [ ]:
# 1. 98th Percentile of Character Length
char_98 = final_train_df['selfies'].str.len().quantile(0.98)
print(f"98% of selfies have fewer than {int(char_98)} characters.")

# 2. 98th Percentile of Token Count (Approximate)
# Counting '[' is a very accurate proxy for token count in SELFIES
token_98 = final_train_df['selfies'].apply(lambda x: x.count('[')).quantile(0.98)
print(f"98% of selfies have fewer than {int(token_98)} tokens.")

# Inference

In [ ]:
import pandas as pd
import torch
import os
from pathlib import Path
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.trainer_utils import get_last_checkpoint
from tqdm.auto import tqdm

# ==========================================
# 1. SETUP & FIND CHECKPOINT
# ==========================================
drive.mount('/content/drive', force_remount=True)

# Point this to the PARENT folder where checkpoints are saved
# (The 'output_dir' from your training config)
CHECKPOINT_DIR = Path("/content/drive/MyDrive/Altegrad/BioT5_Experiment/checkpoints")
OUTPUT_CSV_PATH = Path("/content/drive/MyDrive/Altegrad/BioT5_Experiment/submission.csv")

# Automatically find the latest checkpoint folder
last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))

if last_checkpoint is None:
    raise FileNotFoundError(f"❌ No checkpoints found in {CHECKPOINT_DIR}. Please check the path.")

print(f"✅ Found latest checkpoint: {last_checkpoint}")
MODEL_PATH = last_checkpoint

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

# ==========================================
# 2. FINALIZE YOUR DATAFRAME
# ==========================================
# (Assuming 'all_selfies' and 'all_ids' are populated from your previous snippet)
dataset_df_test = pd.DataFrame({
    "ID": all_ids,
    "selfies": all_selfies
})

print(f"Test Data Ready: {len(dataset_df_test)} samples")

# ==========================================
# 3. LOAD MODEL & TOKENIZER
# ==========================================
print("Loading model from checkpoint...")

# Use local_files_only to ensure we load YOUR trained weights, not the internet's
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH, local_files_only=True)

model.to(device)
model.eval()

# ==========================================
# 4. INFERENCE FUNCTION
# ==========================================
def generate_descriptions(df, batch_size=32):
    descriptions = []

    # Standardize input format
    # IMPORTANT: Ensure <bom> and <eom> are treated correctly by the tokenizer
    inputs_text = [
        f"Molecule description: <bom>{s}<eom>"
        for s in df['selfies']
    ]

    print("Starting generation...")

    for i in tqdm(range(0, len(inputs_text), batch_size)):
        batch_texts = inputs_text[i : i + batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=256,
                num_beams=5,
                early_stopping=True
            )

        # Decode
        batch_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        descriptions.extend([pred.strip() for pred in batch_preds])

    return descriptions

# ==========================================
# 5. RUN & SAVE
# ==========================================
generated_texts = generate_descriptions(dataset_df_test, batch_size=32)

submission_df = pd.DataFrame({
    "ID": dataset_df_test["ID"],
    "description": generated_texts
})

submission_df.to_csv(OUTPUT_CSV_PATH, index=False)

print("✅ Inference complete!")
print(submission_df.head())
print(f"Saved to: {OUTPUT_CSV_PATH}")

In [ ]:
submission_df.to_csv("sub.csv")